# Spaceship Titanic - KNN dan Naive Bayes

**Kelompok 7**

- Aloysius Pijar Hutama Indrianto (24/534591/PA/22675)
- Indratanaya Budiman (24/534784/PA/22683)
- Gilbert Nathaniel (24/533877/PA/22623)
- Pison Golda Mountera (24/543770/PA/23107)
- KOMA

Notebook ini berisi alur final untuk klasifikasi Kaggle Spaceship Titanic. Model akhir dibatasi sesuai aturan tugas, yaitu hanya menggunakan `KNeighborsClassifier` dan/atau varian Naive Bayes dari scikit-learn.

## Ringkasan alur kerja

1. Memuat `train.csv`, `test.csv`, dan `sample_submission.csv` dari folder `data-from-kaggle`.
2. Membuat fitur tambahan dari `PassengerId`, `Cabin`, `Name`, umur, status cryo/VIP, dan pengeluaran fasilitas.
3. Membaca hasil cross-validation dari `reports/experiments.csv` untuk memilih model terbaik berdasarkan `cv_mean` tertinggi dan `cv_std` terendah.
4. Membangun ulang pipeline model terbaik dari kode eksperimen.
5. Melatih model pada seluruh data train, membuat prediksi data test, dan menyimpan file submission final.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd()
if not (ROOT / "src" / "train_knn_nb.py").exists() and (ROOT.parent / "src" / "train_knn_nb.py").exists():
    ROOT = ROOT.parent

SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import train_knn_nb as knnnb

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print(f"Project root: {ROOT}")
print(f"Random state: {knnnb.RANDOM_STATE}")
print(f"CV folds: {knnnb.N_SPLITS}")

Project root: C:\programming\4th-sem\ml\after-uts\tugas-3_hackathon
Random state: 42
CV folds: 5


## Data

Target `Transported` hanya tersedia di train. Data test tidak dipakai untuk membuat fitur berbasis target atau label bocoran; test hanya diprediksi setelah pipeline final dilatih pada seluruh train.

In [2]:
X, y, test, sample_submission = knnnb.load_data()
train_raw = pd.read_csv(knnnb.TRAIN_PATH)

print(f"Train shape: {train_raw.shape}")
print(f"X train shape: {X.shape}")
print(f"Test shape: {test.shape}")
print("Target distribution:")
print(y.value_counts(normalize=True).rename("proportion"))

display(train_raw.head())

Train shape: (8693, 14)
X train shape: (8693, 13)
Test shape: (4277, 13)
Target distribution:
Transported
True     0.503624
False    0.496376
Name: proportion, dtype: float64


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## Feature engineering

Semua fitur dibuat tanpa memakai label test atau target train di dalam proses transformasi:

- `PassengerId` dipisah menjadi nomor grup, nomor penumpang dalam grup, ukuran grup, dan indikator solo.
- `Cabin` dipisah menjadi deck, nomor kabin, sisi kabin, region nomor kabin, dan kombinasi deck-side.
- `Name` dipakai untuk indikator nama tersedia, panjang nama, dan ukuran kelompok surname di data yang sedang ditransformasi.
- Kolom pengeluaran dibuat menjadi total spend, luxury spend, service spend, spend per person, spend per age, share spend, indikator no/any spend, dan indikator per fasilitas.
- `CryoSleep` dan `VIP` diubah menjadi flag diketahui/tidak diketahui serta flag inferensi sederhana yang hanya memakai nilai fitur non-target.
- `Age` dibuat menjadi missing flag, age band, dan indikator child/teen/adult/senior.
- Kategori gabungan seperti `HomeDestination` membantu KNN/NB menangkap interaksi kategorikal tanpa model non-allowed.

In [3]:
feature_preview = knnnb.SpaceshipFeatureEngineer().fit_transform(X.head(8))
selected_preview_cols = [
    "PassengerId", "GroupNumber", "PassengerNumber", "GroupSize", "IsSolo",
    "CabinDeck", "CabinNumber", "CabinSide", "CabinRegion", "DeckSide",
    "TotalSpend", "LuxurySpend", "ServiceSpend", "NoSpend", "AnySpend",
    "Age", "AgeBand", "HomeDestination",
]

display(feature_preview[selected_preview_cols])

,PassengerId,GroupNumber,PassengerNumber,GroupSize,IsSolo,CabinDeck,CabinNumber,CabinSide,CabinRegion,DeckSide,TotalSpend,LuxurySpend,ServiceSpend,NoSpend,AnySpend,Age,AgeBand,HomeDestination
0,0001_01,1,1,1.0,1.0,B,0,P,000-299,B_P,0.0,0.0,0.0,1.0,0.0,39.0,adult,Europa_TRAPPIST-1e
1,0002_01,2,1,1.0,1.0,F,0,S,000-299,F_S,736.0,627.0,702.0,0.0,1.0,24.0,young_adult,Earth_TRAPPIST-1e
2,0003_01,3,1,2.0,0.0,A,0,S,000-299,A_S,10383.0,10340.0,6807.0,0.0,1.0,58.0,older_adult,Europa_TRAPPIST-1e
3,0003_02,3,2,2.0,0.0,A,0,S,000-299,A_S,5176.0,5176.0,3522.0,0.0,1.0,33.0,adult,Europa_TRAPPIST-1e
4,0004_01,4,1,1.0,1.0,F,1,S,000-299,F_S,1091.0,788.0,870.0,0.0,1.0,16.0,teen,Earth_TRAPPIST-1e
5,0005_01,5,1,1.0,1.0,F,0,P,000-299,F_P,774.0,774.0,291.0,0.0,1.0,44.0,adult,Earth_PSO J318.5-22
6,0006_01,6,1,2.0,0.0,F,2,S,000-299,F_S,1584.0,1542.0,42.0,0.0,1.0,26.0,young_adult,Earth_TRAPPIST-1e
7,0006_02,6,2,2.0,0.0,G,0,S,000-299,G_S,0.0,0.0,0.0,1.0,0.0,28.0,young_adult,Earth_TRAPPIST-1e


## Memilih model terbaik dari laporan eksperimen

`reports/experiments.csv` berisi hasil cross-validation yang dibuat oleh script eksperimen. Seleksi model dilakukan dengan mengurutkan `cv_mean` menurun, lalu `cv_std` menaik agar model dengan rata-rata akurasi lebih tinggi dan variasi fold lebih kecil lebih diprioritaskan.

In [4]:
experiments_report = pd.read_csv(knnnb.EXPERIMENTS_PATH)
experiments_report["cv_mean"] = pd.to_numeric(experiments_report["cv_mean"], errors="coerce")
experiments_report["cv_std"] = pd.to_numeric(experiments_report["cv_std"], errors="coerce")
experiments_report = experiments_report.sort_values(["cv_mean", "cv_std"], ascending=[False, True]).reset_index(drop=True)

best_row = experiments_report.iloc[0]
print("Best experiment from reports:")
print(best_row[["name", "family", "batch", "cv_mean", "cv_std", "fold_scores"]].to_string())

display(experiments_report.head(10))

Best experiment from reports:
name           knn_weight_micro_bin070_num125_k23_distance_p1
family                                                    KNN
batch                                        knn_weight_micro
cv_mean                                              0.803519
cv_std                                               0.010463
fold_scores      0.810236;0.798735;0.808511;0.814730;0.785386


,name,family,cv_mean,cv_std,fold_scores,batch,run_at
0,knn_weight_micro_bin070_num125_k23_distance_p1,KNN,0.803519,0.010463,0.810236;0.798735;0.808511;0.814730;0.785386,knn_weight_micro,2026-05-07T23:06:21
1,knn_weight_micro_bin070_num120_k23_distance_p1,KNN,0.803519,0.009155,0.810236;0.798735;0.809086;0.811853;0.787687,knn_weight_micro,2026-05-07T23:06:21
2,knn_weight_fine_cat100_bin075_num125_skew100_k...,KNN,0.803404,0.010542,0.811386;0.798735;0.809086;0.813003;0.784810,knn_weight_fine,2026-05-07T23:02:59
3,knn_weight_micro_bin075_num125_k23_distance_p1,KNN,0.803404,0.010542,0.811386;0.798735;0.809086;0.813003;0.784810,knn_weight_micro,2026-05-07T23:06:21
4,knn_weight_micro_bin080_num125_k23_distance_p1,KNN,0.803174,0.008853,0.808511;0.798735;0.806786;0.813579;0.788262,knn_weight_micro,2026-05-07T23:06:21
5,knn_weight_micro_bin080_num130_k23_distance_p1,KNN,0.803059,0.011115,0.811961;0.795860;0.806210;0.815880;0.785386,knn_weight_micro,2026-05-07T23:06:21
6,knn_weight_micro_bin070_num130_k23_distance_p1,KNN,0.802944,0.010076,0.810236;0.795860;0.806786;0.814730;0.787112,knn_weight_micro,2026-05-07T23:06:21
7,knn_weight_micro_bin080_num120_k23_distance_p1,KNN,0.802944,0.008630,0.809086;0.797010;0.808511;0.811277;0.788838,knn_weight_micro,2026-05-07T23:06:21
8,knn_weight_fine_cat100_bin075_num125_skew110_k...,KNN,0.802829,0.009702,0.812536;0.798160;0.808511;0.808976;0.785961,knn_weight_fine,2026-05-07T23:02:59
9,knn_weight_micro_bin075_num130_k23_distance_p1,KNN,0.802599,0.009381,0.809661;0.796435;0.805635;0.813579;0.787687,knn_weight_micro,2026-05-07T23:06:21


## Pipeline final

Model terbaik pada laporan saat ini adalah KNN. Preprocessing numerik memakai imputasi median, transformasi `log1p` untuk fitur spend yang skewed, scaling, one-hot encoding untuk kategori, dan bobot transformer untuk menyeimbangkan kontribusi grup fitur pada jarak KNN.

Estimator final tetap memenuhi aturan karena prediksi akhir dibuat oleh `sklearn.neighbors.KNeighborsClassifier`.

In [5]:
batch_experiments = knnnb.build_experiments(str(best_row["batch"]))
best_experiment = next(exp for exp in batch_experiments if exp.name == best_row["name"])

print(best_experiment.name)
print(best_experiment.pipeline)

knn_weight_micro_bin070_num125_k23_distance_p1
Pipeline(steps=[('features', SpaceshipFeatureEngineer()),
                ('preprocess',
                 ColumnTransformer(transformer_weights={'binary': 0.7,
                                                        'categorical': 1.0,
                                                        'numeric': 1.25,
                                                        'skewed': 1.0},
                                   transformers=[('skewed',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('log1p',
                                                                   FunctionTransformer(func=<function _clip_log1p at 0x0000015A674371A0>)),
                                                                  ('scaler',
                                       

## Validasi cross-validation

Cell ini menghitung ulang fold accuracy untuk model terpilih dengan `StratifiedKFold`, `shuffle=True`, dan `random_state=42`, sama seperti script. Nilai ini digunakan sebagai validasi lokal; pemilihan final tetap mengacu pada laporan eksperimen yang sudah tersimpan.

In [6]:
cv = StratifiedKFold(n_splits=knnnb.N_SPLITS, shuffle=True, random_state=knnnb.RANDOM_STATE)
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    best_experiment.pipeline.fit(X_train, y_train)
    valid_pred = best_experiment.pipeline.predict(X_valid)
    score = accuracy_score(y_valid, valid_pred)
    fold_scores.append(score)
    print(f"Fold {fold}: {score:.6f}")

print(f"Mean accuracy: {np.mean(fold_scores):.6f}")
print(f"Std accuracy:  {np.std(fold_scores):.6f}")
print(f"Report mean:   {float(best_row['cv_mean']):.6f}")
print(f"Report std:    {float(best_row['cv_std']):.6f}")

Fold 1: 0.810236


Fold 2: 0.798735


Fold 3: 0.808511


Fold 4: 0.814730


Fold 5: 0.785386
Mean accuracy: 0.803519
Std accuracy:  0.010463
Report mean:   0.803519
Report std:    0.010463


## Membuat submission

Pipeline final dilatih ulang pada seluruh data train, lalu dipakai untuk memprediksi seluruh data test. File submission disimpan ke `submissions/submission_best.csv` dengan format `PassengerId` dan `Transported`.

In [7]:
best_experiment.pipeline.fit(X, y)
predictions = best_experiment.pipeline.predict(test).astype(bool)

submission = sample_submission.copy()
if not submission["PassengerId"].equals(test["PassengerId"]):
    raise ValueError("sample_submission PassengerId order does not match test.csv")

submission["Transported"] = predictions
submission = submission[["PassengerId", "Transported"]]
submission["Transported"] = submission["Transported"].astype(bool)

submission_path = knnnb.BEST_SUBMISSION_PATH
submission.to_csv(submission_path, index=False)

print(f"Saved submission to: {submission_path}")
print(submission.dtypes)
display(submission.head())

Saved submission to: C:\programming\4th-sem\ml\after-uts\tugas-3_hackathon\submissions\submission_best.csv
PassengerId    object
Transported      bool
dtype: object


,PassengerId,Transported
0,0013_01,False
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
